## MTH9899 Project — Run Notebook

This notebook orchestrates the full pipeline by calling modules in `src/`.
All logic lives in the modules; this file handles configuration, calling, and visualization.

**Run from the `MTH9899-Project/` directory** so that relative paths resolve correctly:
- Intraday data: `data_intraday/`
- Daily data:    `DailyData/`
- Saved model:   `saved_model/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)

from src.data     import build_date_maps, load_daily_data, INTRADAY_DIR, DAILY_DIR
from src.target   import build_all_targets, add_target_pipeline
from src.features import (build_intraday_features, build_daily_features,
                          attach_daily_prev, normalize_features, run_feature_eda)
from src.train    import (prepare_matrices, train_all_models, select_best_model,
                          save_artifacts, weighted_r2)
from src.predict  import load_artifacts, predict

print('Imports OK')

## 1. Load date maps and data

In [ ]:
date_strings, date_list, prev_date = build_date_maps(INTRADAY_DIR)
print(f'Found {len(date_strings)} trading dates: {date_strings[0]} to {date_strings[-1]}')

daily_all = load_daily_data(DAILY_DIR)
print(f'Daily data: {len(daily_all):,} rows, {daily_all["Date"].nunique()} dates')

## 2. Build target variable

In [ ]:
target_df = build_all_targets(date_strings, INTRADAY_DIR)
print(f'Target built for {target_df["Date"].nunique()} dates, {len(target_df):,} rows')
target_df.head()

## 3. Train / validation split

In [ ]:
val_year   = 2014
train_years = [2010, 2011, 2012, 2013]
dev_years  = train_years + [val_year]

df = target_df.copy()
df['Year'] = (df['Date'] // 10_000).astype(int)
df = df[df['Year'].isin(dev_years)].copy()

train_df = df[df['Year'].isin(train_years)].drop(columns=['Year']).copy()
val_df   = df[df['Year'] == val_year].drop(columns=['Year']).copy()

print(f'Train: {train_df["Date"].nunique()} dates, {len(train_df):,} rows')
print(f'Val:   {val_df["Date"].nunique()} dates,   {len(val_df):,} rows')

## 4. Build features

In [ ]:
dev_dates = [int(d) for d in date_strings if int(d) // 10_000 in dev_years]

# 4a. Raw intraday features
feat_df = build_intraday_features(date_strings, dev_dates, INTRADAY_DIR)
print(f'Intraday features: {feat_df.shape}')

# 4b. Attach previous-day daily data (EST_VOL_prev, MDV_63_prev) and
#     compute VolumeSurprise, VolumeMorningAfternoonRatio, VolatilityAdjustedReturn
feat_df = attach_daily_prev(feat_df, daily_all, prev_date)

# 4c. Daily-only features (ShortTermReversal, Momentum21d, DollarVolTrend)
feat_df = build_daily_features(feat_df, daily_all, date_list, prev_date)
print(f'After daily features: {feat_df.shape}')

In [ ]:
# 4d. Normalize: TS z (per Id, past-only) -> 5 MAD CS winsor -> CS z
feat_df, feature_cols = normalize_features(feat_df)
print(f'Features: {feature_cols}')
print(f'Normalized feature rows: {len(feat_df):,}')

In [ ]:
# Merge features into train/val
left_cols = ['Date', 'Id'] + [c for c in train_df.columns if c not in feat_df.columns]
train_df = train_df[left_cols].merge(feat_df, on=['Date', 'Id'], how='inner')
val_df   = val_df[left_cols].merge(feat_df, on=['Date', 'Id'], how='inner')
print(f'Train with features: {len(train_df):,} rows')
print(f'Val with features:   {len(val_df):,} rows')

## 5. Target normalization pipeline

In [ ]:
train_df = add_target_pipeline(train_df, daily_all, prev_date)
val_df   = add_target_pipeline(val_df,   daily_all, prev_date)
print(f'After target pipeline — train: {len(train_df):,}, val: {len(val_df):,}')
print('Target_model = 5MAD_CS( TS_z( Target / EST_VOL_prev ) )')
train_df[['Date', 'Id', 'Target', 'EST_VOL_prev', 'Target_model', 'sample_weight']].head()

## 6. (Optional) Feature EDA

In [ ]:
# Uncomment to run cross-sectional EDA on raw feature distributions
# eda_summary = run_feature_eda(feat_df)
# display(eda_summary.round(4))

## 7. Feature correlation diagnostics

In [ ]:
from sklearn.linear_model import LinearRegression

CORR_DIAG_MAX_ROWS = 30_000
_n = min(CORR_DIAG_MAX_ROWS, len(train_df))
_sub = train_df.sample(n=_n, random_state=42)
_feat = _sub[feature_cols].astype(float)

corr_p = _feat.corr(method='pearson')

def high_corr_pairs(cmat, cols, thresh=0.75):
    pairs = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = cmat.iloc[i, j]
            if np.isfinite(r) and abs(r) >= thresh:
                pairs.append((cols[i], cols[j], float(r)))
    return sorted(pairs, key=lambda x: -abs(x[2]))

print('Top Pearson pairs |r| >= 0.75:')
hp = high_corr_pairs(corr_p, list(corr_p.columns), 0.75)
print('  (none)' if not hp else '\n'.join(f'  {a} <-> {b}  r={r:+.4f}' for a, b, r in hp))

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_p.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(np.arange(len(feature_cols)))
ax.set_yticks(np.arange(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
ax.set_title(f'Pearson correlation — train subsample n={_n:,}')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 8. Train all models

In [ ]:
X_train, y_train, w_train, X_val, y_val, w_val, scaler = prepare_matrices(
    train_df, val_df, feature_cols
)
print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')

In [ ]:
models, scores, results = train_all_models(X_train, y_train, w_train, X_val, y_val, w_val)

summary_df = pd.DataFrame(
    [(k, v) for k, v in scores.items()],
    columns=['Model', 'Val_weighted_R2']
).sort_values('Val_weighted_R2', ascending=False)
display(summary_df)

In [ ]:
best_name, best_model = select_best_model(models, scores)
print(f'Best model: {best_name}  (val weighted R² = {scores[best_name]:.6f})')

save_artifacts('saved_model', best_model, feature_cols, scaler)
print('Artifacts saved.')

## 9. Feature importance (best model)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_val, y_val, n_repeats=8, random_state=42,
    sample_weight=w_val, scoring='r2', n_jobs=-1,
)
imp_mda = pd.DataFrame({
    'Feature': feature_cols,
    'MDA_importance': perm.importances_mean,
    'MDA_std': perm.importances_std,
}).sort_values('MDA_importance', ascending=False)

display(imp_mda)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_mda['Feature'][::-1], imp_mda['MDA_importance'][::-1],
        xerr=imp_mda['MDA_std'][::-1], capsize=2)
ax.set_xlabel('MDA importance (drop in R² when feature shuffled)')
ax.set_title(f'Feature importance (MDA) — {best_name}')
plt.tight_layout()
plt.show()

## 10. Bin plots: top features vs target

In [ ]:
n_top = min(6, len(feature_cols))
top_features = imp_mda['Feature'].head(n_top).tolist()
val_plot = val_df[feature_cols + ['Target_model']].copy()
val_plot['Pred_model'] = best_model.predict(X_val)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for idx, feat in enumerate(top_features):
    ax = axes[idx]
    val_plot['_bin'] = pd.qcut(val_plot[feat], q=5, labels=False, duplicates='drop')
    binned = val_plot.groupby('_bin', observed=True).agg(
        mean_actual=('Target_model', 'mean'),
        mean_pred=('Pred_model', 'mean'),
        mid_feat=(feat, 'median'),
    ).reset_index()
    ax.plot(binned['mid_feat'], binned['mean_actual'], 'o-', label='Mean Target_model', color='C0')
    ax.plot(binned['mid_feat'], binned['mean_pred'],   's--', label='Mean Pred_model',  color='C1')
    ax.set_xlabel(feat)
    ax.set_ylabel('Mean (model label)')
    ax.set_title(feat)
    ax.legend(loc='best', fontsize=7)
    ax.grid(True, alpha=0.3)
for j in range(len(top_features), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Bin plots: top features vs Target_model (5 bins, validation)', y=1.02)
plt.tight_layout()
plt.show()

## 11. White paper: hyperparameter table and metrics

In [ ]:
# Alternative weight: 1/EST_VOL
w_estvol = 1.0 / val_df['EST_VOL_prev'].replace(0, np.nan)
w_estvol = w_estvol.fillna(w_estvol.median()).values
r2_1estvol = r2_score(y_val, best_model.predict(X_val), sample_weight=w_estvol)

print(f'Val R² (weights=√MDV_63):  {scores[best_name]:.6f}')
print(f'Val R² (weights=1/EST_VOL): {r2_1estvol:.6f}')

# Hyperparameter summary table
hp_rows = []
for name, m in models.items():
    if m is None:
        continue
    hp = {k: v for k, v in m.get_params().items()
          if k in ('alpha', 'l1_ratio', 'max_depth', 'learning_rate',
                   'n_estimators', 'min_samples_leaf', 'hidden_layer_sizes')}
    hp_rows.append({'Model': name, 'Hyperparameters': str(hp), 'Val weighted R²': scores[name]})
display(pd.DataFrame(hp_rows))

## 12. White paper plots

In [ ]:
# Bin plot: 4 prediction quartiles vs mean actual target
val_pred = val_df[['Date']].copy()
val_pred['Pred']          = best_model.predict(X_val)
val_pred['y_actual_plot'] = val_df['Target'].values
val_pred['Pred_bin'] = pd.qcut(val_pred['Pred'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
bin_stats = val_pred.groupby('Pred_bin', observed=True)['y_actual_plot'].agg(['mean','std','count'])
bin_stats['se'] = bin_stats['std'] / np.sqrt(bin_stats['count'])

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(bin_stats))
ax.bar(x, bin_stats['mean'], yerr=bin_stats['se'], capsize=5, color='steelblue', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(bin_stats.index)
ax.set_xlabel('Prediction bin (quartiles)')
ax.set_ylabel('Mean actual return (raw Target)')
ax.set_title('Bin plot: predictions vs target (validation 2014)')
ax.axhline(val_pred['y_actual_plot'].mean(), color='gray', ls='--', alpha=0.7, label='Overall mean')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# 30-day MA of cross-sectional correlation (pred vs target_model)
val_pred['y_fit'] = y_val
daily_corr = (
    val_pred.groupby('Date')
    .apply(lambda g: g['Pred'].corr(g['y_fit']))
    .reset_index(name='corr')
)
daily_corr['corr_ma30'] = daily_corr['corr'].rolling(30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_corr.index, daily_corr['corr'],      alpha=0.5, label='Daily cross-sectional corr')
ax.plot(daily_corr.index, daily_corr['corr_ma30'], color='C1', lw=2, label='30-day MA')
ax.set_xlabel('Trading day index (validation 2014)')
ax.set_ylabel('Correlation (pred vs target)')
ax.set_title('Cross-sectional correlation: predictions vs target (30-day MA)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drift plot: mean target by prediction bin over time (4 bins, ±1 SE bands)
val_drift = val_pred.copy()
val_drift['Bin'] = pd.qcut(val_drift['Pred'], q=4, labels=False, duplicates='drop')
drift_agg = val_drift.groupby(['Date','Bin'])['y_actual_plot'].agg(['mean','std','count']).reset_index()
drift_agg['se'] = drift_agg['std'] / np.sqrt(drift_agg['count'].clip(lower=1))
drift_by_date = drift_agg.pivot(index='Date', columns='Bin', values='mean')
se_by_date    = drift_agg.pivot(index='Date', columns='Bin', values='se')

fig, ax = plt.subplots(figsize=(10, 4))
for b in drift_by_date.columns:
    mean_vals = drift_by_date[b].values
    se_vals   = np.nan_to_num(se_by_date[b].values if b in se_by_date.columns else np.zeros_like(mean_vals))
    x = np.arange(len(mean_vals))
    ax.plot(x, mean_vals, label=f'Bin {b}', alpha=0.9)
    ax.fill_between(x, mean_vals - se_vals, mean_vals + se_vals, alpha=0.2)
ax.set_xlabel('Trading day index (validation 2014)')
ax.set_ylabel('Mean actual return in bin')
ax.set_title('Drift plot: mean target by prediction bin over time (4 bins; ±1 SE)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()